In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
import os
import sys
import glob
from sklearn.utils import shuffle
from pandas.plotting import scatter_matrix
import matplotlib.pyplot as plt
from sklearn import model_selection
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.utils.multiclass import unique_labels

# import ML Classifiers 
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier 
from sklearn.ensemble import AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

In [2]:
#Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile

#File path to the zipped datafile
zip_path = '../../Data/Processed/Data_Compressed.zip'

#List of filenames to read from the zip file
filenames = [
    'Data_Compressed/all_normalized_features.csv',
    'Data_Compressed/kdd_all_normalized_features.csv',
    'Data_Compressed/kdd_expanded_all_scaled.csv',
    'Data_Compressed/kdd_merged_normalized_all.csv'
]

#Corresponding names for the DataFrames
df_names = [
    'xuetangx_df',
    'kdd_df',
    'kdd_expanded_df',
    'kdd_merged_df'
]

#Create an empty dictionary to store the DataFrames
dataframes = {}

#Open the zip file
z = zipfile.ZipFile(zip_path, 'r')

#Loop through the filenames and read them into DataFrames
for i, file in enumerate(filenames):
    print(f"Reading file: {file} from {zip_path}")
    
    #Read each CSV file directly from the zip
    f = z.open(file)
    dataframes[df_names[i]] = pd.read_csv(f)
    f.close()
    
    print(f"{df_names[i]} loaded with shape: {dataframes[df_names[i]].shape}\n")

#Close the zip file after reading
z.close()

#Assign individual DataFrames to variables
xuetangx_df = dataframes['xuetangx_df']
kdd_df = dataframes['kdd_df']
kdd_expanded_df = dataframes['kdd_expanded_df']
kdd_merged_df = dataframes['kdd_merged_df']

Reading file: Data_Compressed/all_normalized_features.csv from ../../Data/Processed/Data_Compressed.zip
xuetangx_df loaded with shape: (225642, 30)

Reading file: Data_Compressed/kdd_all_normalized_features.csv from ../../Data/Processed/Data_Compressed.zip
kdd_df loaded with shape: (200904, 17)

Reading file: Data_Compressed/kdd_expanded_all_scaled.csv from ../../Data/Processed/Data_Compressed.zip
kdd_expanded_df loaded with shape: (120542, 142)

Reading file: Data_Compressed/kdd_merged_normalized_all.csv from ../../Data/Processed/Data_Compressed.zip
kdd_merged_df loaded with shape: (120542, 158)



In [3]:
#Isolate the X and y features for the kdd dataset
kdd_X = kdd_df.drop(columns=['truth'])
kdd_X = kdd_X.drop(columns=['enrollment_id'])
kdd_y = kdd_df['truth']

In [4]:
#Split the kdd dataset into training and testing sets
kdd_X_train, kdd_X_test, kdd_y_train, kdd_y_test = train_test_split(kdd_X, kdd_y, test_size=0.2, random_state=100)

In [5]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import confusion_matrix, accuracy_score

def evaluate_model(model, kdd_X_train, kdd_y_train, kdd_X_test, kdd_y_test, scoring='roc_auc', random_seed=100):
    kfold = KFold(n_splits=5, random_state=random_seed, shuffle=True)
    cv_results = cross_val_score(model, kdd_X_train, kdd_y_train, cv=kfold, scoring=scoring)
    
    model.fit(kdd_X_train, kdd_y_train)
    y_pred = model.predict(kdd_X_test)  # <--- Predict on the test set, not the training set
    
    cm = confusion_matrix(kdd_y_test, y_pred)
    acc = accuracy_score(kdd_y_test, y_pred)
    
    print(f'\nModel: {model.__class__.__name__}')
    print(f'Cross-validation mean score ({scoring}): {cv_results.mean():.4f} ({cv_results.std():.4f})')
    print(f'Accuracy: {acc:.4f}')
    print('Confusion Matrix:')
    print(cm)
    print('-' * 50)

# Define models
models = {
    'Logistic Regression': LogisticRegression(solver='liblinear', multi_class='auto'),
    'LDA': LinearDiscriminantAnalysis(),
    'KNN': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(),
    'Naive Bayes': GaussianNB(),
    'AdaBoost': AdaBoostClassifier(n_estimators=100),
    'Random Forest': RandomForestClassifier(n_estimators=10),
    'XGBoost': XGBClassifier()
}

def run_experiments(kdd_X_train, kdd_y_train, kdd_X_test, kdd_y_test):
    for name, model in models.items():
        evaluate_model(model, kdd_X_train, kdd_y_train, kdd_X_test, kdd_y_test)

# Example usage:
run_experiments(kdd_X_train, kdd_y_train, kdd_X_test, kdd_y_test)


/Users/kirstintretter/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/kirstintretter/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/kirstintretter/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/kirstintretter/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/


Model: LogisticRegression
Cross-validation mean score (roc_auc): 0.8451 (0.0047)
Accuracy: 0.8534
Confusion Matrix:
[[ 3544  4794]
 [ 1096 30747]]
--------------------------------------------------

Model: LinearDiscriminantAnalysis
Cross-validation mean score (roc_auc): 0.8430 (0.0048)
Accuracy: 0.8490
Confusion Matrix:
[[ 3157  5181]
 [  887 30956]]
--------------------------------------------------

Model: KNeighborsClassifier
Cross-validation mean score (roc_auc): 0.7768 (0.0083)
Accuracy: 0.8449
Confusion Matrix:
[[ 4101  4237]
 [ 1996 29847]]
--------------------------------------------------

Model: DecisionTreeClassifier
Cross-validation mean score (roc_auc): 0.6166 (0.0020)
Accuracy: 0.8055
Confusion Matrix:
[[ 4017  4321]
 [ 3496 28347]]
--------------------------------------------------

Model: GaussianNB
Cross-validation mean score (roc_auc): 0.8192 (0.0041)
Accuracy: 0.8473
Confusion Matrix:
[[ 3779  4559]
 [ 1578 30265]]
--------------------------------------------------

/Users/kirstintretter/anaconda3/lib/python3.11/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/Users/kirstintretter/anaconda3/lib/python3.11/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/Users/kirstintretter/anaconda3/lib/python3.11/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/Users/kirstintretter/anaconda3/lib/python3.11/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the S


Model: AdaBoostClassifier
Cross-validation mean score (roc_auc): 0.8475 (0.0043)
Accuracy: 0.8595
Confusion Matrix:
[[ 4213  4125]
 [ 1522 30321]]
--------------------------------------------------

Model: RandomForestClassifier
Cross-validation mean score (roc_auc): 0.8119 (0.0034)
Accuracy: 0.8445
Confusion Matrix:
[[ 4555  3783]
 [ 2464 29379]]
--------------------------------------------------

Model: XGBClassifier
Cross-validation mean score (roc_auc): 0.8470 (0.0043)
Accuracy: 0.8600
Confusion Matrix:
[[ 4121  4217]
 [ 1410 30433]]
--------------------------------------------------
